# 01 · Abrir o lote

Aula **"A imagem como dado"** e o caso do *lote que ninguém abriu*: 4 mil fotos, rótulo
caprichado, e no meio da coleta trocaram a câmera. Antes de qualquer filtro ou rede, olhar o dado:

1. quantas imagens, de que classe, em que conjunto;
2. a foto é uma planilha: **shape, dtype e faixa**;
3. como são os raio-X de cada classe, e a **imagem média** de cada uma;
4. **histograma** e **brilho / contraste / nitidez** por classe: existe atalho que não é o coração?
5. **duplicatas** e **quase duplicatas** entre treino e teste: a base não traz ID de paciente.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from src import config
from src.data.dados import baixar_base, carregar_imagens, dividir, listar_imagens
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import auditoria

configurar_estilo()
SECAO = "01-abrir-o-lote"

## 1. Baixar e catalogar

O zip vem do Zenodo e tem o MD5 conferido. A divisão treino/teste é a da própria base; a
validação sai do treino, estratificada.

In [ ]:
baixar_base()
catalogo = dividir(listar_imagens())

contagem = pd.crosstab(
    catalogo["conjunto"], catalogo["classe"].map(config.ROTULOS), margins=True, margins_name="total"
).reindex(["treino", "validacao", "teste", "total"])
salvar_tabela(contagem, SECAO, "contagem")
contagem

## 2. A foto é uma planilha

Cada raio-X é uma matriz de números: um canal (tons de cinza), inteiros de 0 (preto) a 255
(branco). No raio-X, **osso e coração são claros** (barram o feixe) e **pulmão é escuro** (ar).

In [ ]:
imagens = carregar_imagens(catalogo["caminho"])
classes = catalogo["classe"].to_numpy()
print("shape:", imagens.shape, "| dtype:", imagens.dtype, "| faixa:", imagens.min(), "a", imagens.max())

exemplo = imagens[0]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.imshow(exemplo, cmap="gray", vmin=0, vmax=255)
ax1.add_patch(plt.Rectangle((60, 60), 8, 8, fill=False, ec=config.CORAL, lw=2))
ax1.set_title(f"{catalogo['id'][0]} ({config.ROTULOS[classes[0]]})")
ax1.axis("off")
recorte = exemplo[60:68, 60:68]
ax2.imshow(recorte, cmap="gray", vmin=0, vmax=255)
for (i, j), v in np.ndenumerate(recorte):
    ax2.text(j, i, v, ha="center", va="center", fontsize=8, color="white" if v < 128 else "black")
ax2.set_title("Recorte 8×8: a mesma região em números")
ax2.axis("off")
salvar_figura(fig, SECAO, "foto-e-planilha", pd.DataFrame(recorte))
plt.show()

## 3. Como são as duas classes

Seis exemplos de cada classe e a **imagem média** de cada uma (a média pixel a pixel de
todos os raio-X da classe, só no treino).

In [ ]:
rng = np.random.default_rng(config.SEMENTE)
fig, eixos = plt.subplots(2, 6, figsize=(13, 4.8))
for linha, classe in enumerate([0, 1]):
    escolha = rng.choice(np.where(classes == classe)[0], 6, replace=False)
    for ax, i in zip(eixos[linha], escolha):
        ax.imshow(imagens[i], cmap="gray", vmin=0, vmax=255)
        ax.set_title(catalogo["id"][i], fontsize=8)
        ax.axis("off")
    eixos[linha, 0].text(-12, 64, config.ROTULOS[classe], rotation=90, va="center", ha="right",
                         fontsize=11, fontweight="bold", color=config.CORES_CLASSE[classe])
fig.suptitle("Amostras de treino e teste", fontweight="bold", color=config.AZUL_PROFUNDO)
salvar_figura(fig, SECAO, "amostras")
plt.show()

In [ ]:
treino = (catalogo["conjunto"] == "treino").to_numpy()
media = {k: imagens[treino & (classes == k)].astype(np.float32).mean(axis=0) for k in (0, 1)}
diferenca = media[1] - media[0]

fig, eixos = plt.subplots(1, 3, figsize=(12, 4.3))
for ax, k in zip(eixos, (0, 1)):
    ax.imshow(media[k], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"Média · {config.ROTULOS[k]}")
    ax.axis("off")
lim = np.abs(diferenca).max()
im = eixos[2].imshow(diferenca, cmap="RdBu_r", vmin=-lim, vmax=lim)
eixos[2].set_title("Cardiomegalia − Normal")
eixos[2].axis("off")
fig.colorbar(im, ax=eixos[2], fraction=0.046, label="diferença de intensidade")
salvar_figura(fig, SECAO, "imagem-media", pd.DataFrame(diferenca))
plt.show()

## 4. Histograma e qualidade por classe

Se uma classe for, em média, mais clara, mais contrastada ou mais nítida, a rede pode aprender
**isso** em vez do tamanho do coração. Para medir o tamanho do atalho, cada medida é usada
sozinha como classificador: AUC 0,5 = não separa nada; AUC longe de 0,5 = atalho.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
niveis = np.arange(257)
hist = pd.DataFrame(index=niveis[:-1])
for k in (0, 1):
    h, _ = np.histogram(imagens[classes == k], bins=niveis, density=True)
    hist[config.ROTULOS[k]] = h
    ax.plot(niveis[:-1], h, color=config.CORES_CLASSE[k], label=config.ROTULOS[k], lw=1.8)
ax.set(xlabel="intensidade do pixel", ylabel="proporção de pixels", title="Histograma por classe")
ax.legend()
salvar_figura(fig, SECAO, "histograma-por-classe", hist)
plt.show()

In [ ]:
qualidade = auditoria.qualidade(imagens)
qualidade["classe"] = classes

fig, eixos = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, medida in zip(eixos, ["brilho", "contraste", "nitidez"]):
    for k in (0, 1):
        ax.hist(qualidade.loc[classes == k, medida], bins=40, alpha=0.6,
                color=config.CORES_CLASSE[k], label=config.ROTULOS[k])
    ax.set_title(medida.capitalize())
eixos[0].legend()
salvar_figura(fig, SECAO, "qualidade-por-classe", qualidade)
plt.show()

atalhos = pd.DataFrame({
    "mediana_normal": qualidade[qualidade.classe == 0].median(),
    "mediana_cardiomegalia": qualidade[qualidade.classe == 1].median(),
    "auc_sozinha": {m: roc_auc_score(classes, qualidade[m]) for m in ["brilho", "contraste", "nitidez"]},
}).drop("classe")
salvar_tabela(atalhos, SECAO, "atalhos")
atalhos.round(3)

## 5. Duplicatas exatas

In [ ]:
duplicatas = auditoria.duplicatas_exatas(imagens)
print("Grupos de duplicatas exatas:", len(duplicatas))

## 6. Quase duplicatas entre treino e teste

A base não informa o paciente. O ChestX-ray14 tem vários exames da mesma pessoa, que ficam
muito parecidos. Para cada raio-X de **teste**, procura-se o mais parecido no **treino + validação**
(correlação das imagens reduzidas a 32×32, que ignora brilho e contraste).

In [ ]:
assin = auditoria.assinatura(imagens)
teste = (catalogo["conjunto"] == "teste").to_numpy()
idx_teste, idx_treino = np.where(teste)[0], np.where(~teste)[0]
viz, corr = auditoria.vizinho_mais_proximo(assin[idx_teste], assin[idx_treino])
par = idx_treino[viz]

limiares = [0.90, 0.92, 0.93, 0.95]
resumo = pd.DataFrame({
    "imagens_de_teste": [(corr >= t).sum() for t in limiares],
    "fracao_do_teste": [(corr >= t).mean() for t in limiares],
    "mesmo_rotulo_do_vizinho": [(classes[idx_teste] == classes[par])[corr >= t].mean() for t in limiares],
}, index=pd.Index(limiares, name="correlacao_minima"))
salvar_tabela(resumo, SECAO, "quase-duplicatas")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.hist(corr, bins=60, color=config.AZUL_COBALTO)
ax.axvline(config.LIMIAR_QUASE_DUPLICATA, color=config.CORAL, ls="--",
           label=f"limiar {config.LIMIAR_QUASE_DUPLICATA}")
ax.set(xlabel="correlação com o raio-X de treino mais parecido", ylabel="imagens de teste",
       title="Quão perto cada raio-X de teste está do treino")
ax.legend()
salvar_figura(fig, SECAO, "vizinho-mais-proximo", pd.DataFrame({"correlacao": corr}))
plt.show()
resumo.round(3)

In [ ]:
ordem = np.argsort(-corr)[:8]
fig, eixos = plt.subplots(2, 8, figsize=(15, 4.6))
for col, o in enumerate(ordem):
    for lin, i in enumerate([idx_teste[o], par[o]]):
        ax = eixos[lin, col]
        ax.imshow(imagens[i], cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"{catalogo['id'][i]}\n{config.ROTULOS[classes[i]]}", fontsize=7,
                     color=config.CORES_CLASSE[classes[i]])
        ax.axis("off")
    eixos[0, col].text(64, -38, f"r = {corr[o]:.3f}", ha="center", fontsize=9, fontweight="bold")
fig.suptitle("Os 8 pares mais parecidos (em cima: teste · embaixo: treino)",
             fontweight="bold", color=config.AZUL_PROFUNDO, y=1.06)
salvar_figura(fig, SECAO, "pares-mais-parecidos")
plt.show()

### Calibrar o limiar olhando

Correlação alta não garante mesmo paciente, e correlação um pouco menor não garante pessoa
diferente. Três faixas, oito pares cada (em cima: teste · embaixo: treino):

In [ ]:
ordem_corr = np.argsort(-corr)
faixas = [(0.93, 0.94), (0.915, 0.925), (0.90, 0.905)]
fig, eixos = plt.subplots(6, 8, figsize=(13, 10.5))
for f, (lo, hi) in enumerate(faixas):
    sel = [k for k in ordem_corr if lo <= corr[k] < hi][:8]
    for col, k in enumerate(sel):
        for lin, i in enumerate([idx_teste[k], par[k]]):
            ax = eixos[2 * f + lin, col]
            ax.imshow(imagens[i], cmap="gray", vmin=0, vmax=255)
            ax.axis("off")
    eixos[2 * f, 0].text(-20, 128, f"r entre\n{lo} e {hi}", ha="right", va="center",
                         fontsize=10, fontweight="bold", color=config.AZUL_PROFUNDO)
fig.suptitle("Mesma pessoa? Pares por faixa de correlação", fontweight="bold",
             color=config.AZUL_PROFUNDO)
salvar_figura(fig, SECAO, "calibrar-limiar")
plt.show()

Entre 0,93 e 0,94 quase todos os pares são a mesma pessoa (mesmo marcador, mesma postura,
mesmos fios). Perto de 0,92, mais ou menos metade. Em 0,90 já são pessoas diferentes que
só têm enquadramento parecido. O projeto adota **0,92** (`config.LIMIAR_QUASE_DUPLICATA`):
pega quase todos os pares reais, ao custo de juntar alguns que não são.

## 7. Grupos de imagens parecidas na base inteira

Sem ID de paciente, o substituto é ligar todo par com correlação ≥ limiar (inclusive em cadeia)
num mesmo **grupo**. O notebook 06 usa esses grupos para um split que nunca separa dois
exames quase iguais e mede quanto a acurácia muda.

In [ ]:
grupos = auditoria.grupos_parecidos(assin, config.LIMIAR_QUASE_DUPLICATA)
catalogo["grupo"] = grupos
tamanho = catalogo.groupby("grupo")["id"].transform("size")
cruzam = catalogo.groupby("grupo")["conjunto"].transform(lambda s: s.isin(["teste"]).any() and not s.eq("teste").all())

resumo_grupos = pd.Series({
    "imagens": len(catalogo),
    "grupos": catalogo["grupo"].nunique(),
    "imagens em grupos com 2 ou mais": int((tamanho > 1).sum()),
    "maior grupo": int(tamanho.max()),
    "imagens em grupos que cruzam treino/teste": int(cruzam.sum()),
    "grupos com rótulos misturados": int((catalogo.groupby("grupo")["classe"].nunique() > 1).sum()),
}, name="valor")
salvar_tabela(resumo_grupos.to_frame(), SECAO, "grupos-parecidos")
catalogo[["id", "conjunto", "classe", "grupo"]].to_csv(
    config.DATA_PROCESSED / "grupos.csv", sep=";", index=False)
resumo_grupos

Os grupos se formam **em cadeia**: se A parece com B e B com C, os três ficam juntos mesmo
que A e C não se pareçam. Por isso aparecem alguns grupos grandes. Para o split por grupo isso é
inofensivo (só mantém mais imagens do mesmo lado); para medir o vazamento, a medida honesta é a
da seção 6: quantos raio-X de teste têm um par **direto** no treino.

## Conclusões

* **O lote é limpo no formato:** 5.552 raio-X 128×128, um canal, `uint8` de 0 a 255, classes
  exatamente equilibradas em treino (3.550), validação (888) e teste (1.114). Nenhuma duplicata exata.
* **Não há atalho global de qualidade:** brilho, contraste e nitidez, cada um sozinho, separam as
  classes com AUC entre 0,45 e 0,53, perto do acaso. Não houve "troca de câmera" entre as classes.
* **Mas a imagem média avisa:** a maior diferença entre as classes não está na borda do coração,
  e sim nos **pulmões mais claros** e nas **margens mais escuras** da cardiomegalia. É coerente com
  pacientes mais graves (congestão, filme feito no leito). Uma CNN pode aprender isso em vez do
  tamanho do coração; o notebook 07 (mapas de ativação) verifica.
* **Existe vazamento por paciente, pequeno:** 76 raio-X de teste (6,8 %) têm um par no treino com
  correlação ≥ 0,92, e olhando os pares a maioria é visivelmente a mesma pessoa. O notebook 06
  mede quanto isso infla a acurácia, comparando com um split que respeita os grupos.
* **O rótulo tem ruído:** mesmo entre pares quase idênticos, só 60–70 % têm o mesmo rótulo. Parte
  pode ser evolução real entre exames, mas os rótulos do ChestX-ray14 foram extraídos de laudos
  por processamento de texto, com erro conhecido. Isso põe um **teto** em qualquer acurácia e
  precisa constar como limitação.